In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from gensim.models import Word2Vec, FastText
from tqdm import tqdm
import re


In [14]:
# Загрузка
df_pos = pd.read_csv("/kaggle/input/rutweetcorp/positive.csv", delimiter=';', names=['id', 'date', 'name', 'text', 'typr', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])
df_neg = pd.read_csv("/kaggle/input/rutweetcorp/negative.csv", delimiter=';', names=['id', 'date', 'name', 'text', 'typr', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])


df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[['text', 'typr']]  

# токенизация
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.split()

df["tokens"] = df["text"].apply(simple_tokenize)


In [15]:
# Обучение моделей
sentences = df["tokens"].tolist()

w2v_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4)
ft_model = FastText(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4)

# Функция усреднения
def avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

X_w2v = np.array([avg_vector(t, w2v_model) for t in df['tokens']])
X_ft = np.array([avg_vector(t, ft_model) for t in df['tokens']])
y = (df['typr'] == 1).astype(int).values  # Преобразуем в 0 и 1

print('Done')

Done


In [16]:
class FeedforwardNN(nn.Module):
    def __init__(self, input_size, layer_sizes):
        super().__init__()
        layers = []
        current_size = input_size
        for hidden_size in layer_sizes:
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            current_size = hidden_size
        layers.append(nn.Linear(current_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.network(x))


In [17]:
def train_nn(X, y, layer_sizes, lr, epochs=10):
    model = FeedforwardNN(X.shape[1], layer_sizes)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).view(-1, 1)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_tensor)
        loss = criterion(output, y_tensor)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        preds = model(X_tensor).numpy()
        preds = (preds > 0.5).astype(int)
        acc = accuracy_score(y_tensor, preds)
        f1 = f1_score(y_tensor, preds)
    return acc, f1, loss.item()


In [18]:
results = []
configs = [
    ([32], [0.01, 0.001]),
    ([64, 32], [0.001, 0.0001]),
    ([128, 64, 32], [0.01, 0.0001])
]

for model_name, X_data in [("Word2Vec", X_w2v), ("FastText", X_ft)]:
    for layers, lrs in configs:
        for lr in lrs:
            acc, f1, loss = train_nn(X_data, y, layers, lr)
            results.append({
                "Модель": model_name,
                "Слои": "-".join(map(str, layers)),
                "Скорость обучения": lr,
                "Точность": round(acc, 4),
                "F1": round(f1, 4),
                "Потери": round(loss, 4)
            })

df_results = pd.DataFrame(results)
df_results


,Модель,Слои,Скорость обучения,Точность,F1,Потери
0,Word2Vec,32,0.0100,0.6044,0.5891,0.6634
1,Word2Vec,32,0.0010,0.5713,0.4892,0.6810
2,Word2Vec,64-32,0.0010,0.5547,0.3746,0.6852
3,Word2Vec,64-32,0.0001,0.5068,0.6726,0.6928
4,Word2Vec,128-64-32,0.0100,0.6099,0.6216,0.6654
5,Word2Vec,128-64-32,0.0001,0.4934,0.0000,0.6944
6,FastText,32,0.0100,0.6246,0.6584,0.6463
7,FastText,32,0.0010,0.5943,0.6056,0.6730
8,FastText,64-32,0.0010,0.5778,0.4742,0.6789
9,FastText,64-32,0.0001,0.4974,0.0987,0.6928


Итоговое


In [2]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from gensim.models import Word2Vec, FastText
from tqdm import tqdm
import re

In [3]:
# Загрузка позитивных и негативных твитов из корпуса RuTweetCorp
df_pos = pd.read_csv("/kaggle/input/rutweetcorp/positive.csv", delimiter=';', names=['id', 'date', 'name', 'text', 'typr', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])
df_neg = pd.read_csv("/kaggle/input/rutweetcorp/negative.csv", delimiter=';', names=['id', 'date', 'name', 'text', 'typr', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])

# Объединение позитивных и негативных твитов в один датафрейм
df = pd.concat([df_pos, df_neg], ignore_index=True)

df = df[['text', 'typr']]

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.split()

# Применяем токенизацию к каждому тексту
df["tokens"] = df["text"].apply(simple_tokenize)

# Извлечение списка предложений для обучения моделей Word2Vec и FastText
sentences = df["tokens"].tolist()

In [4]:
# Обучение Word2Vec и FastText на корпусе твитов
w2v_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4)
ft_model = FastText(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4)

# Функция усреднения векторов слов для получения одного вектора на всё сообщение
def avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

# Применение усреднения векторов для всех твитов
X_w2v = np.array([avg_vector(t, w2v_model) for t in df['tokens']])
X_ft = np.array([avg_vector(t, ft_model) for t in df['tokens']])

# Преобразование меток классов: позитив — 1, негатив — 0
y = (df['typr'] == 1).astype(int).values

# Подготовка словаря для nn.Embedding: каждому уникальному слову сопоставляется уникальный индекс
from collections import defaultdict
word2idx = defaultdict(lambda: 0)
word2idx['<PAD>'] = 0  # паддинг = 0
idx = 1
for tokens in df['tokens']:
    for token in tokens:
        if token not in word2idx:
            word2idx[token] = idx
            idx += 1

In [5]:
# Ограничиваем длину каждого сообщения до max_len, добавляем паддинги
max_len = 30
def tokens_to_ids(tokens):
    ids = [word2idx[token] for token in tokens[:max_len]]
    return ids + [0] * (max_len - len(ids))

# Получаем массив индексов слов (для nn.Embedding)
X_emb_ids = np.array([tokens_to_ids(tokens) for tokens in df['tokens']])
vocab_size = len(word2idx)

# Делим данные на обучающую и тестовую выборки (80/20)
X_w2v_train, X_w2v_test, y_train, y_test = train_test_split(X_w2v, y, test_size=0.2, random_state=42)
X_ft_train, X_ft_test = train_test_split(X_ft, test_size=0.2, random_state=42)
X_emb_train, X_emb_test = train_test_split(X_emb_ids, test_size=0.2, random_state=42)


In [6]:
# Класс простой полносвязной нейросети
class FeedforwardNN(nn.Module):
    def __init__(self, input_size, layer_sizes):
        super().__init__()
        layers = []
        current_size = input_size
        for hidden_size in layer_sizes:
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())  # активация ReLU
            current_size = hidden_size
        layers.append(nn.Linear(current_size, 1))  # выходной слой
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.network(x))  # на выходе для вероятности

In [7]:
# Класс модели с nn.Embedding и усреднением векторов слов
class EmbeddingNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, layer_sizes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.feedforward = FeedforwardNN(embed_dim, layer_sizes)

    def forward(self, x):
        embeds = self.embedding(x)          # получаем эмбеддинги
        avg_embed = embeds.mean(dim=1)      # усреднение по всем словам
        return self.feedforward(avg_embed)

In [8]:
# Функция обучения модели и расчёта метрик
def train_nn(X_train, X_test, y_train, y_test, layer_sizes, lr, use_embedding=False):
    if use_embedding:
        model = EmbeddingNN(vocab_size, 100, layer_sizes)
    else:
        model = FeedforwardNN(X_train.shape[1], layer_sizes)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    if use_embedding:
        X_train_tensor = torch.LongTensor(X_train)
        X_test_tensor = torch.LongTensor(X_test)
    else:
        X_train_tensor = torch.FloatTensor(X_train)
        X_test_tensor = torch.FloatTensor(X_test)

    y_train_tensor = torch.FloatTensor(y_train).view(-1, 1)
    y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

    for epoch in range(10):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_tensor)
        loss = criterion(output, y_train_tensor)
        loss.backward()
        optimizer.step()

    # Оценка модели на тестовой выборке
    with torch.no_grad():
        model.eval()
        preds = model(X_test_tensor).numpy()
        preds = (preds > 0.5).astype(int)
        acc = accuracy_score(y_test_tensor, preds)
        f1 = f1_score(y_test_tensor, preds)
    return acc, f1, loss.item()

In [9]:
# Конфигурации сети: число слоёв и скорости обучения
results = []
configs = [
    ([32], [0.01, 0.001, 0.0001]),
    ([64, 32], [0.01, 0.001, 0.0001]),
    ([128, 64, 32], [0.01, 0.001, 0.0001])
]

In [10]:
# Запуск обучения и сбора результатов для всех моделей и конфигураций
for model_name, X_train, X_test, is_emb in [
    ("Word2Vec", X_w2v_train, X_w2v_test, False),
    ("FastText", X_ft_train, X_ft_test, False),
    ("TorchEmbedding", X_emb_train, X_emb_test, True)
]:
    for layers, lrs in configs:
        for lr in lrs:
            acc, f1, loss = train_nn(X_train, X_test, y_train, y_test, layers, lr, use_embedding=is_emb)
            results.append({
                "Модель": model_name,
                "Слои": "-".join(map(str, layers)),
                "Скорость обучения": lr,
                "Точность": round(acc, 4),
                "F1": round(f1, 4),
                "Потери": round(loss, 4)
            })


In [12]:
# Сбор результатов в таблицу и сортировка по F1
df_results = pd.DataFrame(results).sort_values(by="F1", ascending=False)
df_results

,Модель,Слои,Скорость обучения,Точность,F1,Потери
6,Word2Vec,128-64-32,0.0100,0.6110,0.6771,0.6568
8,Word2Vec,128-64-32,0.0001,0.5611,0.6756,0.6914
25,TorchEmbedding,128-64-32,0.0010,0.5180,0.6713,0.6918
22,TorchEmbedding,64-32,0.0010,0.5591,0.6665,0.6909
3,Word2Vec,64-32,0.0100,0.6192,0.6641,0.6589
14,FastText,64-32,0.0001,0.5347,0.6625,0.6912
20,TorchEmbedding,32,0.0001,0.5071,0.6554,0.6928
9,FastText,32,0.0100,0.6260,0.6460,0.6486
21,TorchEmbedding,64-32,0.0100,0.6558,0.6436,0.6034
15,FastText,128-64-32,0.0100,0.6206,0.6367,0.6517
